# Otimização do Modelo Campeão 
### Extreme Gradient Boosting

## Biblioteca / Configuração

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação de dados
import pandas as pd
import numpy as np
import json
import os
import pickle
from datetime import datetime

# Visualização
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Diretórios / Funções internas
from config.paths import *
from config.function_models import *
from config.model_metrics import *

# Modelos / Machine Learning
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (roc_auc_score, precision_recall_curve, average_precision_score, roc_curve, confusion_matrix, classification_report)

# Otimização
import optuna

# Avisos
import warnings
warnings.filterwarnings("ignore")

# Configuração de exibição
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", None)

print("Ambiente Configurado")

Diretórios carregadas com sucesso
Metricas do modelo carregadas com sucesso
Ambiente Configurado


## Parâmetros Globais

In [3]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# garante reprodutibilidade dos experimentos
RANDOM_STATE = 42
# Definindo o número de folds na validação cruzada
CV = 5
# data de execução do notebook (para versionamento/controle)
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
# versão do pipeline/modelo
VERSAO = 'V1 - xgboost'

### Carregamento dos dados 

In [4]:
# Carregar datasets processados
train = pd.read_parquet(PROCESSED_DIR / 'abt01_train_fs.parquet')
test =  pd.read_parquet(PREDICTIONS_DIR / 'abt01_test.parquet')

print(f'Treino: {train.shape}')
print(f'Teste: {test.shape}')

Treino: (831081, 57)
Teste: (389550, 57)


# Avaliação da Performance do melhor modelo

In [5]:
# Separar features e target
X = train.drop(columns=[TARGET])
y = train[TARGET]

# Holdout de validação (para early stopping e escolha de threshold)
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Teste permanece totalmente cego para avaliação final
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print(f"Treino: {X_train.shape} | Validação: {X_valid.shape} | Teste: {X_test.shape}")

Treino: (664864, 56) | Validação: (166217, 56) | Teste: (389550, 56)


In [6]:
# definicao do modelo com hiperparametros padrao robustos para baseline
model = XGBClassifier(
    n_estimators=200,          # numero de arvores
    max_depth=6,               # profundidade maxima das arvores
    learning_rate=0.1,         # taxa de aprendizado
    subsample=0.8,             # amostragem de linhas por arvore
    colsample_bytree=0.8,      # amostragem de colunas por arvore
    eval_metric="logloss",     # metrica de avaliacao interna
    random_state=RANDOM_STATE,
    n_jobs=-1                  # usa todos os nucleos
)

# treino do modelo nos dados de treino
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, gpu_id=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, predictor=None, random_state=42, ...)

### sanity check do baseline

In [7]:
# Avalia modelo baseline no conjunto de teste (AUC, PR-AUC e KS)
# probabilidades da classe positiva
probs_baseline = model.predict_proba(X_test)[:, 1]

# métricas principais
auc = roc_auc_score(y_test, probs_baseline)
pr_auc = average_precision_score(y_test, probs_baseline)

# KS
fpr, tpr, thr = roc_curve(y_test, probs_baseline)
ks = np.max(tpr - fpr)

print(f"AUC ROC (baseline): {auc:.4f}")
print(f"AUC PR  (baseline): {pr_auc:.4f}")
print(f"KS      (baseline): {ks:.4f}")

AUC ROC (baseline): 0.7091
AUC PR  (baseline): 0.4073
KS      (baseline): 0.3047


### tratar desbalanceamento

In [8]:
# Treina XGBoost nativo com balanceamento de classe e early stopping

# calcula peso da classe positiva automaticamente
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Scale_pos_weight: {scale_pos_weight:.2f}")

# cria estrutura nativa do XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
dtest  = xgb.DMatrix(X_test, label=y_test)

# parâmetros robustos para crédito
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'seed': RANDOM_STATE,
    'nthread': -1
}

# treino com early stopping em validação
model_xgb = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1000,
    evals=[(dvalid, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=False
)

print(f"Best iteration: {model_xgb.best_iteration}")

# probabilidades no teste
probs_balanced = model_xgb.predict(dtest)

Scale_pos_weight: 3.31
Best iteration: 474


In [9]:
# Avalia modelo com balanceamento

auc = roc_auc_score(y_test, probs_balanced)
pr_auc = average_precision_score(y_test, probs_balanced)

fpr, tpr, thr = roc_curve(y_test, probs_balanced)
ks = np.max(tpr - fpr)

print(f"AUC ROC (balanced): {auc:.4f}")
print(f"AUC PR  (balanced): {pr_auc:.4f}")
print(f"KS      (balanced): {ks:.4f}")

AUC ROC (balanced): 0.7097
AUC PR  (balanced): 0.4085
KS      (balanced): 0.3047


### Optuna

In [49]:
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'eta': trial.suggest_float('eta', 0.05, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 2),
        'lambda': trial.suggest_float('lambda', 0.1, 5, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 5, log=True),
        'seed': RANDOM_STATE,
        'nthread': -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    aucs = []

    for train_idx, valid_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

        params['scale_pos_weight'] = (y_tr == 0).sum() / (y_tr == 1).sum()

        dtrain_fold = xgb.DMatrix(X_tr, label=y_tr)
        dvalid_fold = xgb.DMatrix(X_va, label=y_va)

        model = xgb.train(
            params=params,
            dtrain=dtrain_fold,
            num_boost_round=500,
            evals=[(dvalid_fold, 'valid')],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        preds = model.predict(dvalid_fold)
        aucs.append(roc_auc_score(y_va, preds))

    return float(np.mean(aucs))

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study.optimize(objective, n_trials=20, show_progress_bar=True)

[I 2026-03-12 09:49:57,088] A new study created in memory with name: no-name-f06e3065-ba4d-4511-90fc-c86d214cd5c6


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-12 09:55:00,191] Trial 0 finished with value: 0.7177162790036201 and parameters: {'max_depth': 7, 'eta': 0.054391233272915736, 'subsample': 0.843054641627782, 'colsample_bytree': 0.7095723413070094, 'min_child_weight': 3, 'gamma': 1.3784733100880417, 'lambda': 1.7514944970926478, 'alpha': 3.201316321634073}. Best is trial 0 with value: 0.7177162790036201.
[I 2026-03-12 09:59:53,713] Trial 1 finished with value: 0.7180701276500883 and parameters: {'max_depth': 5, 'eta': 0.05769676155362311, 'subsample': 0.6678926748960556, 'colsample_bytree': 0.6450428136487152, 'min_child_weight': 7, 'gamma': 1.916204943609359, 'lambda': 0.7966347468355832, 'alpha': 0.0011955759503455112}. Best is trial 1 with value: 0.7180701276500883.
[I 2026-03-12 10:04:04,087] Trial 2 finished with value: 0.7179297452914565 and parameters: {'max_depth': 5, 'eta': 0.08047918806247878, 'subsample': 0.7367437694067407, 'colsample_bytree': 0.8292299799888174, 'min_child_weight': 3, 'gamma': 0.520377304170712

### Treinando o melhores parâmetros

In [11]:
# Treina modelo final com melhores hiperparâmetros encontrados no Optuna

# peso da classe positiva no treino
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

# cria DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
dtest  = xgb.DMatrix(X_test, label=y_test)

# parâmetros finais vindos do Optuna + ajustes fixos
params_final = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'seed': RANDOM_STATE,
    'nthread': -1,
    'max_depth': 4,
    'eta': 0.10740270249041461,
    'subsample': 0.9215542866493176,
    'colsample_bytree': 0.9393123533028884,
    'min_child_weight': 4,
    'gamma': 1.6110157764634296,
    'lambda': 0.19986360547903248,
    'alpha': 0.3191367959369211
}

# treino com early stopping em validação
model_final = xgb.train(
    params=params_final,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=[(dvalid, 'valid')],
    early_stopping_rounds=100,
    verbose_eval=False
)

print(f"Best iteration final: {model_final.best_iteration}")

# probabilidades finais no teste (teste continua cego no treino)
probs_final = model_final.predict(dtest)

Best iteration final: 561


In [12]:
# Avalia modelo final

auc = roc_auc_score(y_test, probs_final)
pr_auc = average_precision_score(y_test, probs_final)

fpr, tpr, thr = roc_curve(y_test, probs_final)
ks = np.max(tpr - fpr)

print(f"AUC ROC (final): {auc:.4f}")
print(f"AUC PR  (final): {pr_auc:.4f}")
print(f"KS      (final): {ks:.4f}")

AUC ROC (final): 0.7099
AUC PR  (final): 0.4082
KS      (final): 0.3059


In [13]:
# Calcula KS e métricas por threshold usando o modelo final
from sklearn.metrics import roc_curve, precision_score, recall_score, f1_score, confusion_matrix

# calcula KS
fpr, tpr, thresholds = roc_curve(y_test, probs_final)
ks_values = tpr - fpr
best_idx = np.argmax(ks_values)

best_threshold = thresholds[best_idx]
best_ks = ks_values[best_idx]

print(f"KS máximo: {best_ks:.4f}")
print(f"Threshold KS: {best_threshold:.4f}")

# tabela operacional de decisão
grid = np.linspace(0.05, 0.80, 16)

rows = []
for thr in grid:
    preds = (probs_final >= thr).astype(int)
    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    f1   = f1_score(y_test, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    rows.append([thr, prec, rec, f1, tp, fp, tn, fn])

metrics_table = pd.DataFrame(
    rows,
    columns=["threshold","precision","recall","f1","TP","FP","TN","FN"]
)

display(metrics_table.sort_values("threshold"))

KS máximo: 0.3059
Threshold KS: 0.4779


,threshold,precision,recall,f1,TP,FP,TN,FN
0,0.05,0.226981,0.999853,0.369972,88216,300434,887,13
1,0.10,0.230951,0.996894,0.375021,87955,292883,8438,274
2,0.15,0.239184,0.987986,0.385131,87169,277274,24047,1060
3,0.20,0.249902,0.971529,0.397546,85717,257285,44036,2512
4,0.25,0.262662,0.945687,0.411133,83437,234222,67099,4792
5,0.30,0.277366,0.909350,0.425077,80231,209029,92292,7998
6,0.35,0.293884,0.862358,0.438374,76085,182810,118511,12144
7,0.40,0.312470,0.803704,0.449989,70910,156024,145297,17319
8,0.45,0.332976,0.731256,0.457589,64518,129244,172077,23711
9,0.50,0.356808,0.645400,0.459553,56943,102647,198674,31286


### Salvando o modelo

In [14]:
# salva o modelo nativo XGBoost
model_final.save_model(MODELS_DIR / "modelo_credito_xgb.json")

In [15]:
# Carregar JSON (mais portátil)
model_json = xgb.Booster()
model_json.load_model(MODELS_DIR / "modelo_credito_xgb.json")